# Data Science Pipeline: Two Applied ML / DL Missions

This notebook is an **exercise template, not a worked solution**. Complete the cells yourself and explain your choices in the markdown cells. The structure follows the supplied *Data Science Pipeline* assignment: question/requirements first, then a code cell.

## Mission goals

| | Mission 1: Machine Learning | Mission 2: Deep Learning |
|---|---|---|
| Problem | Diagnose internal pump leakage from hydraulic sensor measurements | Classify environmental sounds from audio recordings |
| Dataset | [UCI Condition Monitoring of Hydraulic Systems](https://archive.ics.uci.edu/dataset/447/condition+monitoring+of+hydraulic+systems) | [ESC-50 environmental sounds](https://github.com/karolpiczak/ESC-50) |
| Main model | **XGBoost** | **PyTorch 2D CNN on log-Mel spectrograms** |
| Required explanation | **SHAP** for model-level and individual predictions | **LIME** for individual spectrogram predictions |
| Task | Binary classification: no pump leakage vs. leakage | Multiclass classification: 50 sound categories |
| Data type | Real measurements from a controlled hydraulic test rig | Real five-second sound recordings |

**Time limit:** Up to **2 working days for ML**, **2 working days for DL**, and **1 day for review and contingencies**, for **no more than one working week**. Aim for roughly 30-35 hours overall. Keep a short log of the time you spend.

### Required deliverables

1. One runnable, well-explained notebook covering both missions (this file).
2. Data checks and a few meaningful figures per mission.
3. A simple baseline and a stronger model or controlled variant, with suitable metrics.
4. **SHAP analysis** in ML and **LIME analysis** in DL, including limitations.
5. Short conclusions: what worked, what failed, and what you would do next with more time.

### Scope boundaries

- Do **not** build a deployed app, dashboard, API, object detector, or extensive slide deck.
- Limit ML tuning to **20-30 trials maximum**. Limit DL to **two model configurations or one baseline plus one deliberate improvement**.
- Use held-out test data **once for final evaluation**. Do not tune based on test-set results.
- There is **no mandatory target accuracy**: a technically sound evaluation and honest analysis matter more than a high number.
- If a step becomes a time sink, record the issue, take a reasonable simplified route, and continue.

### Environment

Suggested tools: Python, NumPy, Pandas, Matplotlib/Seaborn, Scikit-learn, XGBoost, SHAP, PyTorch, Torchaudio (or Librosa), and LIME. A Colab GPU is useful for DL but not essential for the ML section. Set random seeds and record package versions.

> **Reproducibility reminder:** For every model, explain the unit of prediction and what information would be available when a prediction is made. Avoid data leakage.


In [1]:
# TODO: Import common packages, set the random seed and choose CPU/GPU. Record package versions.
import numpy as np
import pandas as pd
import torch


---
# Mission 1: Machine Learning with XGBoost and SHAP

**Dataset:** [UCI Condition Monitoring of Hydraulic Systems](https://archive.ics.uci.edu/dataset/447/condition+monitoring+of+hydraulic+systems)

This is **measured sensor data**, not a synthetic tabular simulation. The component conditions were deliberately varied on a physical test rig. The dataset has 2,205 operating cycles lasting 60 seconds each. Pressure, temperature, flow, vibration, and motor-power sensors sample at different frequencies.

**Classification target:** Use the *internal pump leakage* field from `profile.txt` (third column, zero-based index 2): `0 = no leakage`, `1 = weak leakage`, `2 = severe leakage`. Create a **binary label**: `0` for no leakage, `1` for weak or severe leakage.

**Important:** This is **current-condition diagnosis** from measured operation, **not a forecast of a future breakdown**. Do not describe it as remaining-useful-life prediction.

**Target feature budget:** Start with about 5-8 physically meaningful sensors; extract a few cycle-level features per sensor. You may expand only if time permits.


## 0. Problem understanding

- Define the engineering problem and the **unit of prediction** (one 60-second operating cycle).
- Explain the meaning of the three pump-leakage conditions.
- State what a correct prediction, false alarm, and missed leakage would mean.
- Identify which measurements are predictors and which fields are **targets/condition annotations**.
- Explain why this experiment cannot establish how far in advance a failure could have been predicted.

**Write your answer below** (about 1-2 short paragraphs).


My goal is to examine the sensory data from 8 different sensor type across 2,200 experiments each lasting a second each.

The goal is then to try and figure out how to predict if there was a leakage and its strength (weak vs severe) from the sensory data



## 1. Data loading

- Download the UCI archive; inspect `profile.txt` and the per-sensor text files.
- Choose a manageable initial sensor subset. Suggested starting point: `PS1`, `PS2`, `PS3` (pressure), `FS1` (flow), `TS1` (temperature), `VS1` (vibration), `EPS1` (motor power). You may adjust this selection.
- Verify that **rows correspond to cycles** for every selected sensor, even though sampling rates and within-cycle lengths differ.
- Load targets separately; create a table linking one row/cycle to its pump-leakage label.
- Record the shapes, units, and approximate memory footprint.

**Resource hint:** The UCI dataset is provided as a compressed download with multiple tab-delimited sensor files. You do not need to load every sensor.


In [2]:
# TODO: Download/load a subset of sensor files and profile.txt; verify cycle alignment.

# Loading data from sensor files and profile.txt

# cooling efficiency sensor
CE_raw = pd.read_csv('data\CE.txt', sep='\t')

# cooling power sensor
CP_raw = pd.read_csv('data\CP.txt', sep='\t')

# motor power sensor
EPS1_raw = pd.read_csv('data\EPS1.txt', sep='\t')

# volume flow sensors
FS1_raw = pd.read_csv('data\FS1.txt', sep='\t')
FS2_raw = pd.read_csv('data\FS2.txt', sep='\t')

# profiles of each experiment
profile_raw = pd.read_csv('data\profile.txt', sep='\t')

# pressure motors
PS1_raw = pd.read_csv('data\PS1.txt', sep='\t')
PS2_raw = pd.read_csv('data\PS2.txt', sep='\t')
PS3_raw = pd.read_csv('data\PS3.txt', sep='\t')
PS4_raw = pd.read_csv('data\PS4.txt', sep='\t')
PS5_raw = pd.read_csv('data\PS5.txt', sep='\t')
PS6_raw = pd.read_csv('data\PS6.txt', sep='\t')

# efficiency factor
SE_raw = pd.read_csv('data\SE.txt', sep='\t')

# tempreture sensors
TS1_raw = pd.read_csv('data\TS1.txt', sep='\t')
TS2_raw = pd.read_csv('data\TS2.txt', sep='\t')
TS3_raw = pd.read_csv('data\TS3.txt', sep='\t')
TS4_raw = pd.read_csv('data\TS4.txt', sep='\t')

# vibration sensor
VS1_raw = pd.read_csv('data\VS1.txt', sep='\t')


## 2. Exploratory Data Analysis (EDA)

- Show class balance for pump leakage (no/weak/severe and the new binary label).
- Inspect a few sensor traces from **different operating conditions**; choose at least one pressure-related and one other sensor.
- Identify suspicious ranges, drift, unusually shaped cycles, and differences across conditions.
- Create **2-3 informative charts**, and explain what each reveals.
- State whether adjacent cycles or changed test-rig settings might make a naive random split overly optimistic.

Ask questions that make sense for hydraulic equipment.


In [ ]:
# TODO: EDA tables, cycle traces, class balance, and 2-3 plots.


## 3. Data cleaning

- Check missing, non-finite, duplicate, or implausible sensor readings.
- Ensure all selected sensor files and the profile labels refer to the **same cycles**.
- Keep cycle boundaries intact. Do not interpolate values **across** different cycles.
- Decide how to handle cycles where the stable-condition flag indicates the rig was not fully stable; justify whether to retain them or treat them as a sensitivity check.
- Report what you removed or changed and why.


In [ ]:
# TODO: Validate and clean cycle-level sensor data without mixing cycles.


## 4. Feature engineering

- Convert the raw within-cycle sensor signals into **one feature row per operating cycle**.
- Start with simple summaries such as mean, standard deviation, min, and max for each sensor. Optionally add one motivated feature such as RMS or a pressure difference.
- Name columns clearly (e.g., `PS1_mean`, `PS1_std`), so SHAP results are interpretable.
- **Never include `profile.txt` condition labels as predictors.** Treat virtual efficiency fields cautiously if they summarize the very conditions you are trying to infer.
- Inspect the final feature matrix and explain the physical meaning of the most useful engineered features.

Aim for a compact, interpretable feature matrix; don't create hundreds of arbitrary features.


In [ ]:
# TODO: Aggregate each selected sensor within each cycle; create X and binary y.


## 5. Data splitting

- Split at the **whole-cycle level**, never at the individual sensor-sample level.
- Use reproducible training, validation, and testing sets, for example **70/15/15**, stratified by the binary label if appropriate.
- Fit imputation, scaling (if used), and other learned transformations **only on training data**.
- Explain the limitation that consecutive rig cycles may be correlated. A block/chronological split is an optional sensitivity check only if class coverage remains adequate.
- Keep the test set untouched until the final evaluation.


In [ ]:
# TODO: Create train/validation/test splits and leak-free preprocessing.


## 6. Model selection

- Train **one very simple baseline** (e.g., a dummy classifier or shallow decision tree) so that model performance has context.
- Train **XGBoost (`XGBClassifier`)** as the principal model.
- Account for class imbalance if observed. Explain the method rather than automatically oversampling.
- Compare using meaningful metrics, especially **recall, precision, F1, and PR-AUC**; optionally report ROC-AUC.
- Explain why false negatives and false positives might have different operational consequences.

Two models are enough for the required scope. A Random Forest is optional, not mandatory.


In [ ]:
# TODO: Fit a baseline and XGBoost; compare validation metrics.


## 7. Model optimization

- Tune XGBoost over a **small and justified search space** (maximum **20-30 trials**).
- Consider tree depth, learning rate, number of estimators, subsampling, and class weighting.
- Select models and any classification threshold using **validation** data, not the test set.
- Save a compact experiment table: configuration, validation metric, training time.
- Stop when improvements become marginal or the time budget is exhausted.


In [ ]:
# TODO: Run a bounded tuning experiment and record the selected configuration.


## 8. Evaluation and comparison

- Evaluate the selected XGBoost model on the **held-out test set** once.
- Report confusion matrix, precision, recall, F1, and PR-AUC (or a precision-recall curve).
- Compare the selected model against the baseline and describe any train/validation/test gaps.
- Investigate a few false positives and false negatives at the cycle level.
- Explain how the decision threshold changes the balance between missed leakage and unnecessary inspections.

Include **1-2 high-value plots**; don't build a dashboard.


In [ ]:
# TODO: One final test evaluation, confusion matrix, PR curve, and error examples.


## 9. Model interpretability: REQUIRED SHAP

Use a tree-compatible SHAP explainer (e.g., `shap.TreeExplainer`) on the trained XGBoost model.

**Required:**
1. A global SHAP summary/beeswarm plot on a manageable held-out sample.
2. One dependence/scatter plot investigating a physically meaningful feature.
3. Two local explanations (e.g., waterfall plots): one correctly diagnosed leaking cycle and one challenging or misclassified case.
4. A short discussion: Are the important features physically plausible? Are correlated sensor summaries difficult to disentangle?

**Critical thinking:** SHAP explains how the **model** used these features. It does **not** establish that a sensor feature caused the leakage.


In [ ]:
# TODO: Required: global SHAP, dependence plot, and two local explanations.


## 10. Summary and recommendations

In approximately **200-350 words**, answer:

- How well can you diagnose pump leakage from these measurements?
- What did feature engineering contribute?
- What did SHAP reveal, and what remains uncertain?
- Which mistakes are most concerning operationally?
- What would be needed to move from **current-condition diagnosis** to **future failure prediction**?
- What would you improve if you had one additional week?


In [ ]:
# TODO: Optional: a compact results table or final plot supporting your summary.


## 11. Presentation (OPTIONAL)

No separate slide deck is required. If useful, prepare a **one-page summary** with the problem, data, best model, key metrics, SHAP insight, and limitations. Do not spend more than 30 minutes on formatting.


In [ ]:
# TODO: Optional: export a minimal table/figure. Otherwise leave empty.


---
# Mission 2: Deep Learning with PyTorch and LIME

**Dataset:** [ESC-50: Environmental Sound Classification](https://github.com/karolpiczak/ESC-50)

ESC-50 has **2,000 five-second recordings in 50 categories** (40 recordings per category). The dataset includes `audio/` recordings and `meta/esc50.csv`, with columns such as `filename`, `fold`, `target`, `category`, and `src_file`.

**Classification task:** Given a complete audio recording, predict **one sound class**. This is **clip classification**, not sound-event detection, source separation, or transcription.

**Required architecture:** A modest **PyTorch CNN** consuming **2D log-Mel spectrograms**. No pretrained audio model is required.

**Scope adjustment:** Attempt all **50 classes**. To debug the pipeline quickly, you may first run a subset of classes. If full ESC-50 proves unrealistic within the time limit, explicitly document a smaller **ESC-10** classification experiment using the dataset's `esc10` metadata flag. Do not silently change the task or make target performance claims.

**Interpretability:** Use **LIME image explanations on spectrograms**. A highlighted spectrogram region is a local model explanation, not proof of an identifiable sound source.


## 0. Problem understanding

- Define the task: assign one label to an environmental audio clip.
- Explain why this is multiclass classification rather than detection of events at specific timestamps.
- Describe the categories, clip length, and limited number of examples per class.
- Define an appropriate evaluation objective; explain why a fixed minimum accuracy is not mandatory.
- In your own words, explain the difference between waveform, spectrogram, and log-Mel spectrogram.

**Write your answer below** (about 1-2 short paragraphs).


In [ ]:
# TODO: Optional: inspect category names and listen to one or two examples.


## 1. Data loading

- Download the official ESC-50 repository/data and load `meta/esc50.csv`.
- Find audio files using their filenames in the metadata; verify paths and file availability.
- Load a few waveforms with **Torchaudio or Librosa** and inspect sample rate, number of channels, and duration.
- Create a class-index mapping and verify class counts.
- Record how much data you are loading and how the labels will map to the model's output neurons.

Follow the dataset license and credit the source in your notebook.


In [ ]:
# TODO: Load ESC-50 metadata and audio examples; examine shapes and labels.


## 2. Exploratory Data Analysis (EDA)

- Check clip duration, sample-rate consistency, class distribution, and the provided cross-validation folds.
- Listen to a handful of contrasting classes and confusingly similar examples.
- Plot **one waveform** and **one log-Mel spectrogram** for an example clip.
- Ask why certain sounds might be confused, and what recording background/noise could contribute.
- Keep the analysis visual and concise: **2-3 figures are enough**.


In [ ]:
# TODO: EDA: audio metadata, a waveform, a spectrogram, and class counts.


## 3. Data cleaning and standardization

- Choose one sample rate and a consistent five-second input duration; document resampling/padding decisions.
- Convert to mono consistently if your loader returns multiple channels.
- Check for unreadable, silent, or unusually loud clips and explain any handling.
- Standardize audio amplitude/log-Mel values without using test-set statistics to fit transformations.
- Preserve each clip's source identity and official fold; do not split related excerpts from the same source across sets.


In [ ]:
# TODO: Implement a reproducible audio loading and preprocessing function.


## 4. Feature engineering / representation

- Convert waveforms to **Mel spectrograms** and then log scale (e.g., via `AmplitudeToDB` or log power with a small epsilon).
- Select and justify parameters such as sample rate, `n_fft`, hop length, and `n_mels` (e.g., 64 or 128).
- Make a consistent tensor for the CNN with shape similar to `(batch, 1, mel_bins, time_frames)`.
- Visualize examples from several categories and explain what horizontal/vertical patterns mean.
- Keep preprocessing **identical** for training, evaluation, and LIME predictions, except for intentional training-only augmentation.


In [ ]:
# TODO: Waveform -> log-Mel transform; verify tensor shapes and visualize examples.


## 5. Data splitting

- Use the **provided ESC-50 folds**. For one fast reproducible run, use folds **1-3 for training**, **4 for validation**, and **5 for testing**.
- Explain that the official folds keep clips from the same original sound source together, reducing leakage.
- Create `Dataset`/`DataLoader` objects for all three splits, and use a fixed seed for reproducibility.
- Do not tune models using the test fold.
- Full five-fold cross-validation is **optional** and not expected under the one-week time limit.


In [ ]:
# TODO: Create fold-based train/validation/test DataLoaders without source leakage.


## 6. Model selection

- Build **one modest PyTorch CNN** with convolution, nonlinearity, pooling, and a final classifier.
- Output **50 logits** for full ESC-50, or **10 logits** only if you explicitly choose the ESC-10 fallback.
- Use a multiclass loss such as `nn.CrossEntropyLoss()` with integer class labels.
- Implement training and validation loops that log loss, accuracy, and macro F1 (or compute F1 at each epoch).
- Save the best checkpoint based on validation performance.
- Briefly justify why a 2D CNN is suitable for time-frequency input.


In [ ]:
# TODO: Define the CNN, loss, optimizer, and train/validation functions.


## 7. Model optimization

- Train the initial CNN and record learning curves.
- Run **one deliberate improvement** (not a broad architecture search). Examples: time/frequency masking, light dropout, or weight decay.
- Compare the two configurations with the **same validation fold** and report train/validation behavior.
- Use early stopping and a sensible maximum epoch budget. Stop tuning when improvement stalls or time is exhausted.
- If you cannot reach high accuracy, explain plausible reasons: only 40 examples per class, background noise, model capacity, and overfitting.


In [ ]:
# TODO: Train baseline and one modified configuration; log/plot learning curves.


## 8. Evaluation and comparison

- Test the selected checkpoint on **fold 5** exactly once.
- Report accuracy, macro F1, and a confusion matrix (or a focused subset of the most confused classes if a 50x50 matrix is unreadable).
- Inspect/listen to at least **four mistakes**; describe possible acoustic causes for confusion.
- Compare validation versus test results and note sensitivity to the small number of examples per class.
- Show **one learning-curve plot** and **one concise error-analysis visualization or table**.


In [ ]:
# TODO: Final held-out evaluation, class-level results, and selected errors.


## 9. Model interpretability: REQUIRED LIME

Use **`lime.lime_image.LimeImageExplainer`** on the **2D log-Mel spectrogram image** for individual clips.

**Required:**
1. Explain **four clips**: two correctly classified and two misclassified (if available).
2. For each, show the original spectrogram and a LIME region overlay/heatmap for the predicted class.
3. Build a small **batch prediction wrapper** that accepts the perturbed NumPy spectrogram images LIME supplies, converts them into the CNN's expected tensor shape, and returns **class probabilities** (not logits).
4. Discuss what the emphasized time-frequency regions might represent, and compare this with what you hear in the clip.
5. State at least two limitations of LIME: explanation sensitivity to perturbations/segmentation, and the possibility that masking spectrogram regions creates unrealistic audio patterns.

**Time cap:** Start with ~**300-500 perturbation samples per clip** and no more than four clips; increase only if the explanations are clearly unusable and time remains.

**Avoid a common bug:** LIME must call the same preprocessing/normalization that your model expects; do not accidentally explain a separately rescaled plotting image as though it were an original model input.


In [ ]:
# TODO: LIME predict_proba wrapper; 4 spectrogram explanations and discussion.


## 10. Summary and recommendations

In approximately **200-350 words**, answer:

- What performance did the chosen CNN achieve, and which classes were difficult?
- Did your one controlled improvement help generalization?
- What did LIME reveal about individual sound classifications?
- Did LIME highlight plausible acoustic regions, or was interpretation uncertain?
- What are the limitations of a small clip-classification dataset?
- What would be the next experiment if you had more time?


In [ ]:
# TODO: Optional: compact final metric table or example collage.


## 11. Presentation (OPTIONAL)

No separate slide deck is required. If useful, prepare a **one-page summary** with an example sound, learning curves, best metrics, one LIME visualization, and major limitations. Limit formatting time to 30 minutes.


In [ ]:
# TODO: Optional: export a figure or results table. Otherwise leave empty.


---
## Final submission checklist

- [ ] Both projects use the requested datasets and clearly define their labels.
- [ ] Data splitting happens at the right unit: **whole sensor cycle** for ML, **source-safe audio fold** for DL.
- [ ] Only training data is used to fit preprocessing parameters.
- [ ] ML contains an XGBoost model, validation comparison, test metrics, and **SHAP** plots.
- [ ] DL contains a trained PyTorch CNN, held-out test metrics, and **LIME** explanations.
- [ ] You discussed **what the explanation techniques do and do not prove**.
- [ ] You documented model errors and limitations, rather than only reporting accuracy.
- [ ] The notebook runs top to bottom in a fresh runtime with accessible dataset paths.
- [ ] The total work is scoped to **five working days at most**.

### Dataset and method references

- [UCI Hydraulic Condition Monitoring dataset](https://archive.ics.uci.edu/dataset/447/condition+monitoring+of+hydraulic+systems)
- [ESC-50 dataset and five-fold metadata](https://github.com/karolpiczak/ESC-50)
- [XGBoost Python API](https://xgboost.readthedocs.io/en/stable/python/python_api.html)
- [SHAP TreeExplainer](https://shap.readthedocs.io/en/stable/generated/shap.TreeExplainer.html)
- [LIME image explainer documentation](https://lime-ml.readthedocs.io/en/latest/lime.html)
